# Exercise 7.9 — Barren Plateau in an Explicit Two-Qubit Circuit

**Chapter 7: Quantum Machine Learning** &nbsp;|&nbsp; *Exercises*

---

The barren-plateau formula is usually met as an asymptotic statement, exponentially small gradients
at large $N$. This exercise pins it down at the smallest size where it is still exact, $N=2$, and
checks the closed form against direct Monte-Carlo sampling of the gradient. The point is not the
number itself but that it can be verified: if your own implementation of the variance formula does
not reproduce $0.107$ here, it is wrong, and you will find that out on two qubits instead of twenty.


## 1. Physical Motivation

A variational circuit is trained by following $\partial C/\partial\theta$. If that derivative is
typically exponentially small, the landscape is flat, no gradient signal survives the shot noise,
and the optimizer wanders. This is the *barren plateau*.

Section 7.3 derives, for a circuit deep enough to form a 2-design,
$$
\mathrm{Var}\bigl[\partial_\theta C\bigr] \;=\; \frac{\Tr[\hat O^2]}{2(D^2-1)}\;\mathbb{E}[F_{ii}],
\qquad \mathbb{E}[F_{ii}] = \frac{D}{D+1},
$$
where the second factor is the Haar-averaged quantum Fisher information for a traceless Pauli
generator. Both factors come from the $k=2$ Weingarten calculus of Chapter 3, and neither is
asymptotic: the formula is exact at every $D\ge2$.


## 2. Mathematical Toolbox

**The circuit.** Two qubits, $D=4$, with the parameter inserted between two independent Haar blocks,
$$
C(\theta) \;=\; \langle 0|\,\hat V_1^\dagger\, e^{+i\theta\hat G/2}\,\hat V_2^\dagger\,\hat O\,\hat V_2\, e^{-i\theta\hat G/2}\,\hat V_1\,|0\rangle .
$$

**The gradient at $\theta=0$.** Differentiating $e^{-i\theta\hat G/2}$ and collecting terms,
$$
\left.\frac{\partial C}{\partial\theta}\right|_{\theta=0}
= -\frac{i}{2}\,\langle\psi_1|\,[\hat G,\;\hat V_2^\dagger \hat O \hat V_2]\,|\psi_1\rangle ,
\qquad |\psi_1\rangle = \hat V_1|0\rangle ,
$$
which is real because the commutator of two Hermitian operators is anti-Hermitian.

**The ingredients for $D=4$, $\hat O = \hat Z\otimes\hat Z$:**

| quantity | value |
|---|---|
| $\Tr[\hat O^2] = \Tr[\hat I_4]$ | $4$ |
| $D^2-1$ | $15$ |
| $\mathbb{E}[F_{ii}] = D/(D+1)$ | $4/5$ |

Note $\Tr[\hat O^2]=4$ and **not** $D=4$ by coincidence: $\hat O=\hat Z\otimes\hat Z$ is a Pauli
string, so $\hat O^2=\hat I$ and the trace is the dimension.


## 3. The closed form

$$
\mathrm{Var}[\partial_\theta C] = \frac{4}{2\cdot 15}\cdot\frac{4}{5} = \frac{16}{150} \approx 0.107 .
$$


In [ ]:
from fractions import Fraction

D = 4                       # two qubits
tr_O2 = Fraction(4)         # Tr[(Z(x)Z)^2] = Tr[I_4] = 4
E_Fii = Fraction(D, D + 1)  # Haar-averaged QFI for a traceless Pauli generator

var_exact = tr_O2 / (2 * (D**2 - 1)) * E_Fii
print(f"Var[dC/dtheta] = {var_exact}  = {float(var_exact):.5f}")
assert var_exact == Fraction(16, 150)

# the large-D shortcut, for comparison
approx = Fraction(4, 2 * D**2)
print(f"large-D approx Tr[O^2]/(2 D^2) = {approx} = {float(approx):.5f}   "
      f"(off by {float(approx/var_exact - 1)*100:.0f}%)")


## 4. Monte-Carlo check

The claim is that $16/150$ is the variance of an actual sampled gradient. We draw $\hat V_1,\hat V_2$
Haar-randomly (a 2-design suffices, and Haar is one), evaluate the gradient in closed form at
$\theta=0$, and take the sample variance.

Sampling Haar unitaries: QR-decompose a complex Ginibre matrix and fix the phases of $R$'s diagonal,
the standard recipe from Chapter 1.


In [ ]:
import numpy as np

rng = np.random.default_rng(2)

Z  = np.array([[1, 0], [0, -1]], dtype=complex)
I2 = np.eye(2, dtype=complex)
O  = np.kron(Z, Z)                 # global observable
D  = 4

def haar(n, rng):
    """Haar-random U(n): QR of a Ginibre matrix with the R-diagonal phase fix (Ch. 1)."""
    A = (rng.normal(size=(n, n)) + 1j * rng.normal(size=(n, n))) / np.sqrt(2)
    Q, R = np.linalg.qr(A)
    return Q * (np.diag(R) / np.abs(np.diag(R)))

def gradient_sample(G, rng):
    """dC/dtheta at theta=0 for one draw of (V1, V2)."""
    V1, V2 = haar(D, rng), haar(D, rng)
    psi = V1 @ np.eye(D)[:, 0]          # |psi_1> = V1|00>
    Oh  = V2.conj().T @ O @ V2          # back-propagated observable
    comm = G @ Oh - Oh @ G
    return (-0.5j * (psi.conj() @ comm @ psi)).real

SAMPLES = 400_000
for name, G in [("Z(x)I", np.kron(Z, I2)), ("I(x)Z", np.kron(I2, Z))]:
    g = np.array([gradient_sample(G, rng) for _ in range(SAMPLES)])
    print(f"generator {name}:  mean = {g.mean():+.5f}   Var = {g.var():.5f}   "
          f"(exact {float(var_exact):.5f})")
    assert abs(g.mean()) < 0.01,                       "gradient must be mean-zero"
    assert np.isclose(g.var(), float(var_exact), rtol=0.02), "variance must match the closed form"

print("\nOK: sampled variance reproduces 16/150 for both generators.")


The mean vanishes and the variance matches. **The mean-zero part matters as much as the
variance:** a barren plateau is not a bias, it is the absence of any signal at all, so the optimizer
sees noise around zero rather than a small but consistent slope.


## 5. Why the plateau is a failure of the *observable*, not the state

At $D=4$ the numbers are unremarkable, $\mathrm{Var}\approx0.107$, perfectly trainable. The
exponential suppression appears in how $\Tr[\hat O^2]$ scales.

- **Global Pauli** $\hat O = \hat Z^{\otimes N}$: $\hat O^2=\hat I$, so $\Tr[\hat O^2]=D$ and
  $\mathrm{Var}\sim 1/(2D) = 2^{-(N+1)}$.
- **Global projector** $\hat O = |0\rangle\!\langle0|^{\otimes N}$: $\Tr[\hat O^2]=1$, so
  $\mathrm{Var}\sim 1/(2D^2)$, suppressed twice as fast.
- **Local Pauli** on $k$ qubits: the effective dimension is $2^k$, not $2^N$.

Meanwhile the Haar-averaged QFI is $\mathbb{E}[F_{ii}] = D/(D+1)\to1$: the state moves an
$\mathcal{O}(1)$ Fubini–Study distance under a parameter shift. The motion is there. A fixed global
observable simply resolves an ever-smaller fraction of it.


In [ ]:
print(f"{'N':>3} {'D':>6} {'global Pauli':>14} {'global projector':>18}")
for N in range(1, 13):
    Dn = 2**N
    var_pauli = Dn / (2 * (Dn**2 - 1)) * (Dn / (Dn + 1))   # Tr[O^2] = D
    var_proj  = 1  / (2 * (Dn**2 - 1)) * (Dn / (Dn + 1))   # Tr[O^2] = 1
    print(f"{N:>3} {Dn:>6} {var_pauli:>14.3e} {var_proj:>18.3e}")

print("\nQFI stays O(1) throughout:")
for N in [1, 4, 8, 12]:
    Dn = 2**N
    print(f"  N={N:>2}:  E[F_ii] = D/(D+1) = {Dn/(Dn+1):.6f}")


## Conclusion

$\mathrm{Var}[\partial_\theta C] = 16/150 \approx 0.107$ is exact at $D=4$ and is reproduced by
direct sampling to within Monte-Carlo error. The formula is not asymptotic, and the two qubits are
enough to test any implementation of it.

The scaling table shows where the plateau comes from: not from the state failing to move, since
$\mathbb{E}[F_{ii}]\to1$, but from a global observable resolving a vanishing fraction of that motion.
The cure is therefore to change the observable, or to stop the circuit short of the 2-design limit,
which is the subject of the rest of Section 7.3.
